# [PyTorch] 여러 모델 간을 동적으로 전환하는 슈퍼 라우터 구현
## ~ Gumbel-Softmax를 통한 완벽한 하드 라우팅 실험 ~

이 노트북에서는 여러 개의 독립적인 특수 모델(프랑스어와 독일어에 특화된 모델 1, 스페인어에 특화된 모델 2)을 통합하고 적절한 모델에 입력을 전달하는 **수퍼 라우터**를 구축합니다.

일반적인 Softmax 기반 라우팅(Soft Routing)에서 발생하는 "Lazy Routing" 문제를 시연하고, 이를 해결하기 위해 **Gumbel-Softmax**를 통해 달성한 완벽한 Hard Routing(깨끗한 100% 대 0% 분할)의 위력을 확인합니다.

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
try:
    from sra_language_models import MoESRALanguageModel
except ImportError:
    from src.sra_language_models import MoESRALanguageModel

torch.manual_seed(42)
random.seed(42)

### 1. 데이터 세트 및 유틸리티 준비
우리는 프랑스어, 독일어, 스페인어를 영어로 번역하는 매우 간단한 단어 수준 데이터세트를 준비합니다.

In [ ]:
PARALLEL_DATA = [
    {"eng": "hello", "fra": "bonjour", "deu": "hallo", "spa": "hola"},
    {"eng": "apple", "fra": "pomme", "deu": "apfel", "spa": "manzana"},
    {"eng": "cat", "fra": "chat", "deu": "katze", "spa": "gato"},
    {"eng": "dog", "fra": "chien", "deu": "hund", "spa": "perro"},
    {"eng": "water", "fra": "eau", "deu": "wasser", "spa": "agua"}
]

vocab = ["[PAD]", "[ENG]", "[FRA]", "[DEU]", "[SPA]", "[SEP]", "[EOS]"]
for pair in PARALLEL_DATA:
    for word in pair.values():
        if word not in vocab:
            vocab.append(word)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

def encode(lang_tag, src_word, tgt_word=None):
    prompt = [word2idx[lang_tag], word2idx[src_word], word2idx["[SEP]"], word2idx["[ENG]"]]
    if tgt_word:
        target = [word2idx[tgt_word], word2idx["[EOS]"]]
        return prompt + target
    return prompt

def get_batch(langs, batch_size=16):
    x = torch.zeros((batch_size, 6), dtype=torch.long)
    y = torch.full((batch_size, 6), -100, dtype=torch.long)
    for i in range(batch_size):
        lang = random.choice(langs)
        lang_tag = f"[{lang.upper()}]"
        pair = random.choice(PARALLEL_DATA)
        seq = encode(lang_tag, pair[lang], pair["eng"])
        x[i, :5] = torch.tensor(seq[:5])
        y[i, 3:5] = torch.tensor(seq[4:6])
    return x, y

### 2. 학습 및 테스트 기능 정의
라우터의 동작(각 언어가 어떤 모델로 라우팅될 확률)을 시각화하는 테스트 기능을 준비합니다.

In [ ]:
def load_balance_loss(router_logits):
    if not router_logits: return torch.tensor(0.0)
    loss = 0.0
    for logits in router_logits:
        probs = F.softmax(logits, dim=-1)
        mean_probs = probs.mean(dim=(0, 1))
        loss += (mean_probs ** 2).sum() * logits.size(-1)
    return loss

def train_model(model, langs, steps=400, lr=2e-3):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    model.train()
    for step in range(steps):
        x, y = get_batch(langs)
        logits, router_logits = model(x)
        ce_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=-100)
        lb_loss = load_balance_loss(router_logits)
        loss = ce_loss + 0.1 * lb_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Trained on {langs} - Final Loss: {loss.item():.4f}")

def test_model(model, langs_to_test, is_super=False):
    model.eval()
    with torch.no_grad():
        for lang in langs_to_test:
            lang_tag = f"[{lang.upper()}]"
            pair = PARALLEL_DATA[0] # Test with 'hello'
            src_word = pair[lang]
            expected = pair["eng"]
            x = torch.tensor([encode(lang_tag, src_word)]).long()
            logits, router_logits = model(x)
            pred_idx = logits[0, 3].argmax().item()
            pred_word = idx2word[pred_idx]
            success = pred_word == expected
            status = "OK" if success else "NG"

            if is_super and router_logits:
                # At test time, display the deterministic probabilities
                top_idx = router_logits[0][0].argmax().item()
                top_model = top_idx + 1
                probs = F.softmax(router_logits[0][0], dim=-1)
                print(f"{status} {lang.upper()} -> ENG: '{src_word}' -> '{pred_word}' | [Routed to Model {top_model}] probs: {[round(p, 3) for p in probs[0].tolist()]}")
            else:
                print(f"{status} {lang.upper()} -> ENG: '{src_word}' -> '{pred_word}'")

### 3. 독립적인 특수 모델 사전 학습(모델 1, 모델 2)
우리는 두 개의 작은 SRA 기반 언어 모델을 준비하고 각각 독립적으로 훈련합니다.
- **모델 1**: 프랑스어 및 독일어 번역 전문
- **모델 2**: 스페인어 번역 전문

In [ ]:
DIM = 64
model1 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=2, k=1, syn_hidden=128)
model2 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=1, k=1, syn_hidden=128)

print("Training Model 1 (fra, deu)...")
train_model(model1, ["fra", "deu"], steps=600)

print("Training Model 2 (spa)...")
train_model(model2, ["spa"], steps=600)

### 4. 슈퍼 라우터 설계 및 구현
모델 1과 모델 2를 캡슐화하고 컨텍스트(전체 입력 문장)를 기반으로 어떤 모델이 처리를 처리해야 하는지 결정하는 Super Router 클래스를 만듭니다.
비교를 위해 세 가지 라우팅 전략을 구현합니다.

1. **`soft`**: 소프트 라우팅. 확률을 가중치로 직접 사용합니다.
2. **`ste`**: 직선 추정기. Top-1을 강제로 선택하지만 Softmax를 사용하는 것처럼 그라데이션이 흐르도록 합니다.
3. **`gumbel`**: Gumbel-Softmax. 훈련 중에만 미분 가능한 하드 라우팅을 수행하기 위해 Gumbel 노이즈를 추가합니다.

In [ ]:
class SuperRouterModel(nn.Module):
    def __init__(self, model1, model2, vocab_size, dim, routing_type='soft'):
        super().__init__()
        self.model1 = model1
        self.model2 = model2
        self.routing_type = routing_type

        # Freeze the base models' weights (do not train them)
        for p in self.model1.parameters(): p.requires_grad = False
        for p in self.model2.parameters(): p.requires_grad = False

        # Embedding layer and discriminator for the router
        self.super_embed = nn.Embedding(vocab_size, dim)
        self.super_key = nn.Linear(dim, 2, bias=False)

    def forward(self, x):
        h = self.super_embed(x)
        # Extract sentence-level features via mean pooling
        h_pool = h.mean(dim=1, keepdim=True) # (B, 1, dim)
        router_logits = self.super_key(h_pool) # (B, 1, 2)

        if self.routing_type == 'soft':
            routing_weights = F.softmax(router_logits, dim=-1)

        elif self.routing_type == 'gumbel':
            if self.training:
                routing_weights = F.gumbel_softmax(router_logits, tau=1.0, hard=True, dim=-1)
            else:
                top_idx = router_logits.argmax(dim=-1, keepdim=True)
                routing_weights = torch.zeros_like(router_logits).scatter_(-1, top_idx, 1.0)

        elif self.routing_type == 'ste':
            probs = F.softmax(router_logits, dim=-1)
            top_idx = probs.argmax(dim=-1, keepdim=True)
            hard_weights = torch.zeros_like(probs).scatter_(-1, top_idx, 1.0)
            if self.training:
                routing_weights = hard_weights.detach() - probs.detach() + probs
            else:
                routing_weights = hard_weights

        out1, _ = self.model1(x)
        out2, _ = self.model2(x)

        w1 = routing_weights[..., 0:1]
        w2 = routing_weights[..., 1:2]
        final_out = out1 * w1 + out2 * w2

        return final_out, [router_logits]

### 5. 실험 및 결과 비교
각 라우팅 전략으로 슈퍼 라우터를 훈련시키고 확률 분포(신뢰도)가 어떻게 변하는지 관찰합니다.

In [ ]:
for routing_type in ['soft', 'ste', 'gumbel']:
    print(f"\n{'='*50}\nTesting Routing Type: {routing_type.upper()}\n{'='*50}")
    torch.manual_seed(42)
    super_model = SuperRouterModel(model1, model2, VOCAB_SIZE, DIM, routing_type=routing_type)

    # Train only the Super Router
    train_model(super_model, ["fra", "deu", "spa"], steps=1000, lr=5e-3)

    # Check the test results (probability distribution)
    test_model(super_model, ["fra", "deu", "spa"], is_super=True)

### 결론
- **SOFT**는 게으르다(`0.9 : 0.1`와 같은 잔여 확률을 남기므로 계산이 여전히 불필요한 모델로 흘러갑니다).
- **STE**는 올바른 모델을 선택하면 학습을 중단하는 경향이 있으며 확률은 `0.6 : 0.4`처럼 불투명하게 유지됩니다.
- **GUMBEL**을 사용하면 노이즈로 인한 견고성이 시작되어 **완벽한 `1.0 : 0.0` 희소 라우팅(불필요한 모델에 대한 계산을 100% 줄임)**을 달성합니다!